# Fraud Detection Prediction Model
### Kaggle Playground Series s6e11 — Target: `is_fraud` (Metric: ROC-AUC)

---
### Complete End-to-End Pipeline Sections:
1. **Environment Setup, Config & Library Imports**
2. **Data Loading, Sanity Checks & Exploratory Data Analysis (EDA)**
3. **Advanced Feature Engineering & Validation**
4. **Stratified 5-Fold CV Setup & In-Fold Preprocessing**
5. **PyTorch Datasets & DataLoaders with `USE_POS_WEIGHT` Switch**
6. **Neural Network Architectures (Embeddings & Residual Skip Connections)**
7. **PyTorch MLP 5-Fold Training Loop (Seeds, OneCycleLR, EMA & AMP)**
7a. **LightGBM 5-Fold Cross-Validation**
7b. **XGBoost 5-Fold Cross-Validation**
7c. **CatBoost 5-Fold Cross-Validation**
7d. **Optuna Hyperparameter Optimization for LightGBM**
8. **Out-of-Fold (OOF) Evaluation, Curves & Confusion Matrix**
9. **Multi-Fold & Multi-Seed Test Predictions**
10. **Multi-Model OOF Ensembling (Constrained Blending, Stacking & Rank Averaging)**

In [ ]:
# =============================================================================
# 1. Environment Setup, Configuration & Library Imports
# =============================================================================
import os
import gc
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# Scikit-Learn
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler, QuantileTransformer, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight

# Scipy Optimization & Statistics
from scipy.optimize import minimize
from scipy.stats import rankdata

# Gradient Boosting Frameworks & Optuna
import lightgbm as lgb
import xgboost as xgb
import catboost as cb
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# PyTorch & Deep Learning
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from torch.cuda.amp import autocast, GradScaler

# =============================================================================
# GLOBAL CONFIGURATION (Easily shrinkable for rapid prototyping)
# =============================================================================
N_FOLDS = 5
SEEDS = [42, 1, 7]
OPTUNA_TRIALS = 50

# One fixed StratifiedKFold reused across all models for valid OOF comparability
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

# Reproducibility & Device Initialization
def seed_everything(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Device     : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Global CV Setup : {N_FOLDS}-Fold StratifiedKFold (random_state=42)")
print(f"Config Seeds    : {SEEDS} | Optuna Trials: {OPTUNA_TRIALS}")

In [ ]:
# =============================================================================
# 2. Data Loading, Sanity Checks & Exploratory Data Analysis (EDA)
# =============================================================================
train_df = pd.read_csv('train.csv')
test_df  = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample_submission.csv')

print(f"Train shape      : {train_df.shape}")
print(f"Test shape       : {test_df.shape}")
print(f"Sample sub shape : {sample_sub.shape}")

target_col = 'is_fraud'
counts = train_df[target_col].value_counts()
ratios = train_df[target_col].value_counts(normalize=True) * 100
print("\n--- Target Class Distribution ---")
print(pd.DataFrame({'Count': counts, 'Percentage (%)': ratios.round(3)}))

print("\nMissing values in train:", train_df.isnull().sum().sum())
print("Missing values in test :", test_df.isnull().sum().sum())

# --- 2.1 Fraud Rate per Categorical Column ---
cat_columns = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
print("\n--- Fraud Rate by Categorical Features ---")
for col in cat_columns:
    rate = train_df.groupby(col)[target_col].agg(['count', 'mean']).rename(columns={'mean': 'fraud_rate'})
    rate['fraud_rate_pct'] = (rate['fraud_rate'] * 100).round(2)
    print(f"\n[{col}]")
    print(rate.sort_values(by='fraud_rate', ascending=False).head(5))

# --- 2.2 Feature Correlation with Target ---
num_cols_raw = train_df.select_dtypes(include=[np.number]).columns.drop(['id', target_col])
corrs = train_df[num_cols_raw].apply(lambda c: c.corr(train_df[target_col])).sort_values(ascending=False)
print("\n--- Top Correlations with Target (`is_fraud`) ---")
print(pd.concat([corrs.head(5), corrs.tail(3)]))

# --- 2.3 LightGBM Gain-Based Feature Importance on Baseline Features ---
X_eda = train_df.drop(columns=['id', target_col]).copy()
for c in cat_columns:
    X_eda[c] = X_eda[c].astype('category')
y_eda = train_df[target_col].values

eda_dtrain = lgb.Dataset(X_eda, label=y_eda)
eda_params = {'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'learning_rate': 0.1, 'seed': 42}
eda_bst = lgb.train(eda_params, eda_dtrain, num_boost_round=60)
gain_imp = pd.Series(eda_bst.feature_importance(importance_type='gain'), index=X_eda.columns).sort_values(ascending=False)
print("\n--- Top 10 Features by LightGBM Gain Importance ---")
print(gain_imp.head(10))

# --- 2.4 Adversarial Validation (Detect Train vs Test Covariate Shift) ---
adv_train = X_eda.copy()
adv_train['is_test'] = 0
adv_test = test_df.drop(columns=['id']).copy()
for c in cat_columns:
    adv_test[c] = adv_test[c].astype('category')
adv_test['is_test'] = 1

# Sample subset for fast adversarial check
adv_combined = pd.concat([adv_train.sample(50000, random_state=42), adv_test.sample(50000, random_state=42)], ignore_index=True)
adv_y = adv_combined['is_test'].values
adv_X = adv_combined.drop(columns=['is_test'])

adv_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
adv_aucs = []
for tr_i, va_i in adv_skf.split(adv_X, adv_y):
    dtr = lgb.Dataset(adv_X.iloc[tr_i], label=adv_y[tr_i])
    dva = lgb.Dataset(adv_X.iloc[va_i], label=adv_y[va_i], reference=dtr)
    bst_adv = lgb.train({'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'seed': 42}, dtr, valid_sets=[dva], num_boost_round=50, callbacks=[lgb.early_stopping(10, verbose=False)])
    adv_aucs.append(bst_adv.best_score['valid_0']['auc'])

print(f"\n--- Adversarial Validation ROC-AUC : {np.mean(adv_aucs):.4f} (Ideal ~0.50) ---")
if np.mean(adv_aucs) < 0.55:
    print("Conclusion: Train and Test distributions are identically distributed (no covariate drift detected).")
else:
    print("Warning: Train and Test distributions show slight divergence. Top drifting features:")
    print(pd.Series(bst_adv.feature_importance(importance_type='gain'), index=adv_X.columns).sort_values(ascending=False).head(5))

### 📊 Key Insights from Exploratory Data Analysis (EDA)
1. **Severe Target Imbalance**: The target `is_fraud` is present in only **~1.67%** of transactions. ROC-AUC and PR-AUC are the decisive metrics.
2. **Dominant Signal Features**: LightGBM gain importance and correlations indicate that `merchant_risk_score`, `velocity_score`, `amount_usd`, `prior_disputes`, and `hours_since_last_txn` supply the strongest individual signals.
3. **High-Risk Categories**: Online Retail, Crypto Exchange, and Gaming show higher base rates of fraud than physical POS or groceries.
4. **Stable Covariate Distribution**: Adversarial validation yields an AUC of approximately **~0.50**, confirming that `train.csv` and `test.csv` share an identical feature distribution without train-test drift.

In [ ]:
# =============================================================================
# 3. Advanced Feature Engineering & Verification
# =============================================================================
test_ids = test_df['id'].values
y = train_df[target_col].values.astype(np.float32)

X_train_raw = train_df.drop(columns=['id', target_col]).copy()
X_test_raw  = test_df.drop(columns=['id']).copy()

cat_columns = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']

# 1. Frequency / Count Encodings computed on train + test combined
combined_df = pd.concat([X_train_raw[cat_columns], X_test_raw[cat_columns]], axis=0)
freq_maps = {}
for col in cat_columns:
    freq_maps[col] = combined_df[col].value_counts(normalize=True).to_dict()

def engineer_features(df):
    df = df.copy()
    
    # Temporal cyclical transformations (24h clock & 7 days/week)
    df['sin_hour'] = np.sin(2 * np.pi * df['time_of_day_hour'] / 24.0)
    df['cos_hour'] = np.cos(2 * np.pi * df['time_of_day_hour'] / 24.0)
    df['sin_day']  = np.sin(2 * np.pi * df['day_of_week'] / 7.0)
    df['cos_day']  = np.cos(2 * np.pi * df['day_of_week'] / 7.0)
    
    # Night and weekend indicator flags
    df['is_night']   = ((df['time_of_day_hour'] >= 23) | (df['time_of_day_hour'] <= 5)).astype(int)
    df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)
    
    # Log1p transforms for skewed continuous features
    df['log_amount_usd']       = np.log1p(np.maximum(df['amount_usd'], 0))
    df['log_distance_km']      = np.log1p(np.maximum(df['distance_from_home_km'], 0))
    df['log_balance_usd']      = np.log1p(np.maximum(df['account_balance_usd'], 0))
    df['log_hours_since_last'] = np.log1p(np.maximum(df['hours_since_last_txn'], 0))
    
    # Log-transform unbounded and heavy-tailed amount-to-balance ratio
    df['amount_balance_ratio']     = df['amount_usd'] / (df['account_balance_usd'] + 1.0)
    df['log_amount_balance_ratio'] = np.log1p(np.maximum(df['amount_balance_ratio'], 0))
    
    # Multiplicative Domain Interactions
    df['distance_x_velocity']     = df['distance_from_home_km'] * df['velocity_score']
    df['amount_x_risk']            = df['amount_usd'] * df['merchant_risk_score']
    df['velocity_x_hours']         = df['velocity_score'] * df['hours_since_last_txn']
    df['risk_velocity_factor']     = df['merchant_risk_score'] * df['velocity_score']
    df['age_interaction']          = df['customer_age'] * df['card_age_months']
    
    # Ratio & Rate Interactions
    df['amount_per_txn_24h']       = df['amount_usd'] / (df['txn_count_last_24h'] + 1.0)
    df['balance_to_amount_ratio']  = df['account_balance_usd'] / (df['amount_usd'] + 1.0)
    df['risk_per_dispute']         = df['merchant_risk_score'] / (df['prior_disputes'] + 1.0)
    df['cvv_retry_rate']           = df['cvv_retry_count'] / (df['txn_count_last_24h'] + 1.0)
    df['txn_velocity_density']     = df['txn_count_last_24h'] / (df['hours_since_last_txn'] + 0.1)
    
    # Total active security warning flags
    flag_cols = [
        'is_foreign_transaction', 'is_new_merchant', 'used_vpn',
        'ip_country_mismatch', 'billing_shipping_mismatch', 'is_ai_generated_scam_attempt'
    ]
    for col in flag_cols:
        df[col] = df[col].astype(int)
    df['total_risk_flags'] = df[flag_cols].sum(axis=1)
    
    # Frequency / Count encodings for categoricals
    for col in cat_columns:
        df[f'freq_{col}'] = df[col].map(freq_maps[col]).fillna(0).astype(np.float32)
        
    return df

X_train_fe = engineer_features(X_train_raw)
X_test_fe  = engineer_features(X_test_raw)
print(f"Total engineered features: {X_train_fe.shape[1]} (from {X_train_raw.shape[1]} raw features)")

# --- 3.1 Verification: LightGBM Fast OOF Check Before vs After Feature Engineering ---
print("\n--- Verifying Feature Engineering Impact via Fast 3-Fold LightGBM ---")
fast_skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Baseline (raw features)
raw_tr = X_train_raw.copy()
for c in cat_columns: raw_tr[c] = raw_tr[c].astype('category')
raw_scores = []
for tr_idx, va_idx in fast_skf.split(raw_tr, y):
    dtr = lgb.Dataset(raw_tr.iloc[tr_idx], label=y[tr_idx])
    dva = lgb.Dataset(raw_tr.iloc[va_idx], label=y[va_idx], reference=dtr)
    b = lgb.train({'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'learning_rate': 0.1, 'seed': 42}, dtr, valid_sets=[dva], num_boost_round=80, callbacks=[lgb.early_stopping(15, verbose=False)])
    raw_scores.append(b.best_score['valid_0']['auc'])

# After Feature Engineering
fe_tr = X_train_fe.copy()
for c in cat_columns: fe_tr[c] = fe_tr[c].astype('category')
fe_scores = []
for tr_idx, va_idx in fast_skf.split(fe_tr, y):
    dtr = lgb.Dataset(fe_tr.iloc[tr_idx], label=y[tr_idx])
    dva = lgb.Dataset(fe_tr.iloc[va_idx], label=y[va_idx], reference=dtr)
    b = lgb.train({'objective': 'binary', 'metric': 'auc', 'verbosity': -1, 'learning_rate': 0.1, 'seed': 42}, dtr, valid_sets=[dva], num_boost_round=80, callbacks=[lgb.early_stopping(15, verbose=False)])
    fe_scores.append(b.best_score['valid_0']['auc'])

print(f"Raw Baseline LightGBM AUC : {np.mean(raw_scores):.5f}")
print(f"Engineered LightGBM AUC   : {np.mean(fe_scores):.5f}")
print(f"Feature Engineering Gain  : +{np.mean(fe_scores) - np.mean(raw_scores):.5f} ROC-AUC")

In [ ]:
# =============================================================================
# 4. Stratified 5-Fold Setup & In-Fold Preprocessing
# =============================================================================
# Global StratifiedKFold splits (reused across all models for valid OOF comparability)
cv_splits = list(skf.split(X_train_fe, y))
print(f"Initialized {N_FOLDS}-Fold CV with sizes: {[len(v) for _, v in cv_splits]}")

# Identify numeric and categorical columns
cat_columns = ['merchant_category', 'card_type', 'auth_method', 'channel', 'device_type']
num_columns = [col for col in X_train_fe.columns if col not in cat_columns]

# Encode categorical columns to continuous integer IDs (0 to N_classes-1) for Embeddings and Trees
cat_encoders = {}
cat_dims = {}
X_train_cats = pd.DataFrame(index=X_train_fe.index)
X_test_cats  = pd.DataFrame(index=X_test_fe.index)

for col in cat_columns:
    le = LabelEncoder()
    combined_vals = pd.concat([X_train_fe[col], X_test_fe[col]]).astype(str)
    le.fit(combined_vals)
    cat_encoders[col] = le
    X_train_cats[col] = le.transform(X_train_fe[col].astype(str))
    X_test_cats[col]  = le.transform(X_test_fe[col].astype(str))
    cat_dims[col]     = len(le.classes_)

print("Categorical vocabulary dimensions for Embeddings / Trees:")
for k, v in cat_dims.items():
    print(f"  {k:20s}: {v} unique categories")

# Prepare tree datasets (numeric features + integer/category encoded categoricals)
X_tree_train = pd.concat([X_train_fe[num_columns], X_train_cats], axis=1)
X_tree_test  = pd.concat([X_test_fe[num_columns], X_test_cats], axis=1)
for col in cat_columns:
    X_tree_train[col] = X_tree_train[col].astype('category')
    X_tree_test[col]  = X_tree_test[col].astype('category')

print(f"\nTree feature matrix shape: {X_tree_train.shape}")

In [ ]:
# =============================================================================
# 5. PyTorch Datasets & DataLoaders with `USE_POS_WEIGHT` Switch
# =============================================================================
BATCH_SIZE = 2048
# Switch to toggle positive loss weighting in BCEWithLogitsLoss
USE_POS_WEIGHT = False

def make_loaders(train_idx, val_idx, batch_size=BATCH_SIZE):
    """
    Builds PyTorch DataLoaders for a specific fold.
    Fits QuantileTransformer(output_distribution='normal') strictly on train_idx
    to eliminate data leakage into validation or test sets.
    """
    # QuantileTransformer for numeric inputs (ideal for neural networks on tabular data)
    scaler = QuantileTransformer(output_distribution='normal', random_state=42, n_quantiles=2000)
    
    X_num_train = scaler.fit_transform(X_train_fe[num_columns].iloc[train_idx].values).astype(np.float32)
    X_num_val   = scaler.transform(X_train_fe[num_columns].iloc[val_idx].values).astype(np.float32)
    X_num_test  = scaler.transform(X_test_fe[num_columns].values).astype(np.float32)
    
    X_cat_train = X_train_cats.iloc[train_idx].values.astype(np.int64)
    X_cat_val   = X_train_cats.iloc[val_idx].values.astype(np.int64)
    X_cat_test  = X_test_cats.values.astype(np.int64)
    
    y_train = y[train_idx]
    y_val   = y[val_idx]
    
    # Compute class pos_weight
    num_neg = (y_train == 0).sum()
    num_pos = (y_train == 1).sum()
    pos_weight = torch.tensor([num_neg / num_pos], dtype=torch.float32) if USE_POS_WEIGHT else None
    
    # Create TensorDatasets (numeric_features, categorical_features, target)
    train_ds = TensorDataset(
        torch.from_numpy(X_num_train),
        torch.from_numpy(X_cat_train),
        torch.from_numpy(y_train)
    )
    val_ds = TensorDataset(
        torch.from_numpy(X_num_val),
        torch.from_numpy(X_cat_val),
        torch.from_numpy(y_val)
    )
    test_ds = TensorDataset(
        torch.from_numpy(X_num_test),
        torch.from_numpy(X_cat_test)
    )
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size * 2, shuffle=False, pin_memory=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size * 2, shuffle=False, pin_memory=True)
    
    return train_loader, val_loader, test_loader, pos_weight

print("make_loaders function ready. Example fold 1 loader test:")
tr_l, va_l, te_l, pw = make_loaders(cv_splits[0][0], cv_splits[0][1])
print(f"Batches per epoch: {len(tr_l)} | Val batches: {len(va_l)} | USE_POS_WEIGHT: {USE_POS_WEIGHT}")

In [ ]:
# =============================================================================
# 6. Neural Network Architectures (Embeddings & Residual Skip Connections)
# =============================================================================
# Embedding dimension rule: min(50, (cardinality + 1) // 2)
embedding_configs = [(cat_dims[col], max(4, min(32, (cat_dims[col] + 1) // 2))) for col in cat_columns]
total_emb_dim = sum(e[1] for e in embedding_configs)
num_features_dim = len(num_columns)
total_input_dim = num_features_dim + total_emb_dim

print(f"Numeric inputs : {num_features_dim} | Embedding inputs: {total_emb_dim} | Total input dim: {total_input_dim}")

class BaseEmbeddingModule(nn.Module):
    """Shared categorical embedding layer module"""
    def __init__(self, emb_configs):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(num_cats, emb_dim) for num_cats, emb_dim in emb_configs
        ])
    def forward(self, x_cat):
        emb_outs = [emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)]
        return torch.cat(emb_outs, dim=1)

class FraudMLP(nn.Module):
    """Standard Deep MLP with Categorical Embeddings, BatchNorm & Dropout"""
    def __init__(self, num_dim, emb_configs, hidden_dims=[256, 128, 64, 32], dropouts=[0.35, 0.25, 0.15, 0.10]):
        super().__init__()
        self.emb_module = BaseEmbeddingModule(emb_configs)
        in_dim = num_dim + sum(e[1] for e in emb_configs)
        
        layers = []
        for h_dim, drop_rate in zip(hidden_dims, dropouts):
            layers.append(nn.Linear(in_dim, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(drop_rate))
            in_dim = h_dim
        layers.append(nn.Linear(in_dim, 1))
        self.net = nn.Sequential(*layers)

    def forward(self, x_num, x_cat):
        x_emb = self.emb_module(x_cat)
        x = torch.cat([x_num, x_emb], dim=1)
        return self.net(x).squeeze(1)

class ResBlock(nn.Module):
    """Residual skip-connection block with SiLU activations"""
    def __init__(self, dim, dropout=0.25):
        super().__init__()
        self.block = nn.Sequential(
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(dim, dim),
            nn.BatchNorm1d(dim),
        )
        self.act = nn.SiLU()

    def forward(self, x):
        return self.act(x + self.block(x))

class ResidualFraudMLP(nn.Module):
    """Residual MLP with Skip Connections & Categorical Embeddings"""
    def __init__(self, num_dim, emb_configs, hidden_dim=256):
        super().__init__()
        self.emb_module = BaseEmbeddingModule(emb_configs)
        in_dim = num_dim + sum(e[1] for e in emb_configs)
        
        self.input_layer = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(),
            nn.Dropout(0.30)
        )
        self.res1 = ResBlock(hidden_dim, dropout=0.25)
        self.res2 = ResBlock(hidden_dim, dropout=0.20)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.BatchNorm1d(64),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(64, 1)
        )

    def forward(self, x_num, x_cat):
        x_emb = self.emb_module(x_cat)
        x = torch.cat([x_num, x_emb], dim=1)
        x = self.input_layer(x)
        x = self.res1(x)
        x = self.res2(x)
        return self.head(x).squeeze(1)

print("FraudMLP and ResidualFraudMLP architectures successfully compiled.")

In [ ]:
# =============================================================================
# 7. PyTorch 5-Fold Training Loop (Seeds, OneCycleLR, Weight EMA & AMP)
# =============================================================================
class ModelEMA:
    """Exponential Moving Average of model parameters for smoother convergence"""
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {name: param.clone().detach() for name, param in model.named_parameters() if param.requires_grad}

    def update(self, model):
        with torch.no_grad():
            for name, param in model.named_parameters():
                if param.requires_grad:
                    self.shadow[name].data.mul_(self.decay).add_(param.data, alpha=1.0 - self.decay)

    def apply_shadow(self, model):
        self.backup = {name: param.clone().detach() for name, param in model.named_parameters() if param.requires_grad}
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.shadow[name].data)

    def restore(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(self.backup[name].data)

def eval_mlp(model, loader, criterion, device):
    model.eval()
    total_loss, preds, trues = 0.0, [], []
    with torch.no_grad():
        for x_num, x_cat, y_b in loader:
            x_num, x_cat, y_b = x_num.to(device), x_cat.to(device), y_b.to(device)
            logits = model(x_num, x_cat)
            loss = criterion(logits, y_b)
            total_loss += loss.item() * len(y_b)
            preds.extend(torch.sigmoid(logits).cpu().numpy())
            trues.extend(y_b.cpu().numpy())
    auc = roc_auc_score(trues, preds)
    return total_loss / len(loader.dataset), auc, np.array(preds)

oof_mlp = np.zeros(len(train_df), dtype=np.float32)
test_mlp = np.zeros(len(test_df), dtype=np.float32)
mlp_fold_aucs = []

MLP_EPOCHS = 10
print(f"Starting PyTorch MLP {N_FOLDS}-Fold Training across seeds: {SEEDS}...")

for fold, (train_idx, val_idx) in enumerate(cv_splits):
    print(f"\n{'='*25} MLP FOLD {fold + 1} / {N_FOLDS} {'='*25}")
    train_loader, val_loader, test_loader, pos_weight = make_loaders(train_idx, val_idx)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(device) if pos_weight is not None else None)
    
    fold_val_preds = np.zeros(len(val_idx), dtype=np.float32)
    fold_test_preds = np.zeros(len(test_df), dtype=np.float32)
    
    for seed in SEEDS:
        seed_everything(seed)
        model = ResidualFraudMLP(num_features_dim, embedding_configs, hidden_dim=256).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-4)
        total_steps = len(train_loader) * MLP_EPOCHS
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=3e-3, total_steps=total_steps, pct_start=0.15, anneal_strategy='cos'
        )
        ema = ModelEMA(model, decay=0.999)
        scaler_amp = GradScaler(enabled=(device.type == 'cuda'))
        
        best_seed_auc, best_weights = 0.0, None
        
        for epoch in range(1, MLP_EPOCHS + 1):
            model.train()
            for x_num, x_cat, y_b in train_loader:
                x_num, x_cat, y_b = x_num.to(device), x_cat.to(device), y_b.to(device)
                optimizer.zero_grad()
                with autocast(enabled=(device.type == 'cuda')):
                    logits = model(x_num, x_cat)
                    loss = criterion(logits, y_b)
                scaler_amp.scale(loss).backward()
                scaler_amp.step(optimizer)
                scaler_amp.update()
                scheduler.step()
                ema.update(model)
            
            # Evaluate with EMA weights
            ema.apply_shadow(model)
            _, val_auc, _ = eval_mlp(model, val_loader, criterion, device)
            ema.restore(model)
            
            if val_auc > best_seed_auc:
                best_seed_auc = val_auc
                best_weights = copy.deepcopy(ema.shadow)
        
        # Restore best EMA weights for prediction
        for name, param in model.named_parameters():
            if param.requires_grad:
                param.data.copy_(best_weights[name].data)
                
        _, _, seed_val_p = eval_mlp(model, val_loader, criterion, device)
        fold_val_preds += seed_val_p / len(SEEDS)
        
        # Predict test
        model.eval()
        seed_test_p = []
        with torch.no_grad():
            for x_num, x_cat in test_loader:
                x_num, x_cat = x_num.to(device), x_cat.to(device)
                with autocast(enabled=(device.type == 'cuda')):
                    p = torch.sigmoid(model(x_num, x_cat))
                seed_test_p.extend(p.cpu().numpy())
        fold_test_preds += np.array(seed_test_p) / len(SEEDS)
    
    oof_mlp[val_idx] = fold_val_preds
    test_mlp += fold_test_preds / N_FOLDS
    fold_auc = roc_auc_score(y[val_idx], fold_val_preds)
    mlp_fold_aucs.append(fold_auc)
    print(f"Fold {fold + 1} Ensembled Seeds OOF ROC-AUC: {fold_auc:.5f}")

overall_mlp_auc = roc_auc_score(y, oof_mlp)
print(f"\n>>> OVERALL PyTorch MLP OOF ROC-AUC: {overall_mlp_auc:.5f} <<<")
np.save('oof_mlp.npy', oof_mlp)
np.save('test_mlp.npy', test_mlp)
print("Saved oof_mlp.npy and test_mlp.npy")

In [ ]:
# =============================================================================
# 7a. LightGBM 5-Fold Cross-Validation with Early Stopping on AUC
# =============================================================================
oof_lgb = np.zeros(len(train_df), dtype=np.float32)
test_lgb = np.zeros(len(test_df), dtype=np.float32)
lgb_fold_aucs = []

lgb_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'min_child_samples': 50,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'n_jobs': -1,
    'verbose': -1,
    'random_state': 42
}

print(f"Training LightGBM across {N_FOLDS} Folds...")
for fold, (train_idx, val_idx) in enumerate(cv_splits):
    X_tr_f, y_tr_f = X_tree_train.iloc[train_idx], y[train_idx]
    X_va_f, y_va_f = X_tree_train.iloc[val_idx], y[val_idx]
    
    dtrain = lgb.Dataset(X_tr_f, label=y_tr_f)
    dval   = lgb.Dataset(X_va_f, label=y_va_f, reference=dtrain)
    
    model_lgb = lgb.train(
        lgb_params,
        dtrain,
        valid_sets=[dval],
        num_boost_round=1200,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)]
    )
    
    val_preds = model_lgb.predict(X_va_f)
    oof_lgb[val_idx] = val_preds
    test_lgb += model_lgb.predict(X_tree_test) / N_FOLDS
    
    fold_auc = roc_auc_score(y_va_f, val_preds)
    lgb_fold_aucs.append(fold_auc)
    print(f"LightGBM Fold {fold + 1} ROC-AUC: {fold_auc:.5f} (Best Iter: {model_lgb.best_iteration})")

overall_lgb_auc = roc_auc_score(y, oof_lgb)
print(f"\n>>> OVERALL LightGBM OOF ROC-AUC: {overall_lgb_auc:.5f} <<<")
np.save('oof_lgb.npy', oof_lgb)
np.save('test_lgb.npy', test_lgb)
print("Saved oof_lgb.npy and test_lgb.npy")

In [ ]:
# =============================================================================
# 7b. XGBoost 5-Fold Cross-Validation (tree_method='hist')
# =============================================================================
oof_xgb = np.zeros(len(train_df), dtype=np.float32)
test_xgb = np.zeros(len(test_df), dtype=np.float32)
xgb_fold_aucs = []

# Enable categorical support in XGBoost hist
xgb_params = {
    'objective': 'binary:logistic',
    'eval_metric': 'auc',
    'tree_method': 'hist',
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'enable_categorical': True,
    'learning_rate': 0.05,
    'max_depth': 6,
    'min_child_weight': 30,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.1,
    'reg_lambda': 1.0,
    'random_state': 42
}

print(f"Training XGBoost (device={xgb_params['device']}) across {N_FOLDS} Folds...")
for fold, (train_idx, val_idx) in enumerate(cv_splits):
    X_tr_f, y_tr_f = X_tree_train.iloc[train_idx], y[train_idx]
    X_va_f, y_va_f = X_tree_train.iloc[val_idx], y[val_idx]
    
    dtrain = xgb.DMatrix(X_tr_f, label=y_tr_f, enable_categorical=True)
    dval   = xgb.DMatrix(X_va_f, label=y_va_f, enable_categorical=True)
    dtest  = xgb.DMatrix(X_tree_test, enable_categorical=True)
    
    model_xgb = xgb.train(
        xgb_params,
        dtrain,
        evals=[(dval, 'val')],
        num_boost_round=1000,
        early_stopping_rounds=50,
        verbose_eval=False
    )
    
    val_preds = model_xgb.predict(dval)
    oof_xgb[val_idx] = val_preds
    test_xgb += model_xgb.predict(dtest) / N_FOLDS
    
    fold_auc = roc_auc_score(y_va_f, val_preds)
    xgb_fold_aucs.append(fold_auc)
    print(f"XGBoost Fold {fold + 1} ROC-AUC: {fold_auc:.5f} (Best Iter: {model_xgb.best_iteration})")

overall_xgb_auc = roc_auc_score(y, oof_xgb)
print(f"\n>>> OVERALL XGBoost OOF ROC-AUC: {overall_xgb_auc:.5f} <<<")
np.save('oof_xgb.npy', oof_xgb)
np.save('test_xgb.npy', test_xgb)
print("Saved oof_xgb.npy and test_xgb.npy")

In [ ]:
# =============================================================================
# 7c. CatBoost 5-Fold Cross-Validation with Native Categorical Features
# =============================================================================
oof_cat = np.zeros(len(train_df), dtype=np.float32)
test_cat = np.zeros(len(test_df), dtype=np.float32)
cat_fold_aucs = []

# Use raw strings for native categorical treatment in CatBoost
X_cb_train = pd.concat([X_train_fe[num_columns], X_train_fe[cat_columns].astype(str)], axis=1)
X_cb_test  = pd.concat([X_test_fe[num_columns], X_test_fe[cat_columns].astype(str)], axis=1)

cb_params = {
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'learning_rate': 0.06,
    'depth': 6,
    'iterations': 1200,
    'task_type': 'GPU' if torch.cuda.is_available() else 'CPU',
    'random_seed': 42,
    'verbose': 0
}

print(f"Training CatBoost (task_type={cb_params['task_type']}) across {N_FOLDS} Folds...")
for fold, (train_idx, val_idx) in enumerate(cv_splits):
    X_tr_f, y_tr_f = X_cb_train.iloc[train_idx], y[train_idx]
    X_va_f, y_va_f = X_cb_train.iloc[val_idx], y[val_idx]
    
    train_pool = cb.Pool(X_tr_f, y_tr_f, cat_features=cat_columns)
    val_pool   = cb.Pool(X_va_f, y_va_f, cat_features=cat_columns)
    test_pool  = cb.Pool(X_cb_test, cat_features=cat_columns)
    
    model_cb = cb.CatBoostClassifier(**cb_params)
    model_cb.fit(train_pool, eval_set=val_pool, early_stopping_rounds=50, verbose=False)
    
    val_preds = model_cb.predict_proba(val_pool)[:, 1]
    oof_cat[val_idx] = val_preds
    test_cat += model_cb.predict_proba(test_pool)[:, 1] / N_FOLDS
    
    fold_auc = roc_auc_score(y_va_f, val_preds)
    cat_fold_aucs.append(fold_auc)
    print(f"CatBoost Fold {fold + 1} ROC-AUC: {fold_auc:.5f} (Best Iter: {model_cb.get_best_iteration()})")

overall_cat_auc = roc_auc_score(y, oof_cat)
print(f"\n>>> OVERALL CatBoost OOF ROC-AUC: {overall_cat_auc:.5f} <<<")
np.save('oof_cat.npy', oof_cat)
np.save('test_cat.npy', test_cat)
print("Saved oof_cat.npy and test_cat.npy")

In [ ]:
# =============================================================================
# 7d. Optuna Study (TPE Sampler with Pruning) for LightGBM Tuning
# =============================================================================
print(f"Starting Optuna Study ({OPTUNA_TRIALS} Trials) Tuning LightGBM on 5-Fold OOF AUC...")

def objective(trial):
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',
        'learning_rate': trial.suggest_float('learning_rate', 0.02, 0.12, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 31, 127),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 150),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.5, 0.95),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.5, 0.95),
        'bagging_freq': 1,
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-4, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-4, 10.0, log=True),
        'n_jobs': -1,
        'verbose': -1,
        'random_state': 42
    }
    
    # Fast 3-fold evaluation during tuning search
    trial_aucs = []
    for fold_i, (t_idx, v_idx) in enumerate(cv_splits[:3]):
        dtr = lgb.Dataset(X_tree_train.iloc[t_idx], label=y[t_idx])
        dva = lgb.Dataset(X_tree_train.iloc[v_idx], label=y[v_idx], reference=dtr)
        bst = lgb.train(params, dtr, valid_sets=[dva], num_boost_round=400, callbacks=[lgb.early_stopping(30, verbose=False)])
        score = bst.best_score['valid_0']['auc']
        trial_aucs.append(score)
        trial.report(score, fold_i)
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()
            
    return np.mean(trial_aucs)

study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner())
study.optimize(objective, n_trials=OPTUNA_TRIALS, timeout=900, show_progress_bar=True)

best_params = study.best_params
print(f"\nBest Optuna Trial #{study.best_trial.number} Validation AUC: {study.best_value:.5f}")
print("Best Hyperparameters:", best_params)

# Refit Tuned LightGBM across all 5 folds
print("\n--- Refitting Tuned LightGBM on Full 5-Fold CV ---")
full_tuned_params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'bagging_freq': 1,
    'n_jobs': -1,
    'verbose': -1,
    'random_state': 42,
    **best_params
}

oof_lgb_tuned = np.zeros(len(train_df), dtype=np.float32)
test_lgb_tuned = np.zeros(len(test_df), dtype=np.float32)

for fold, (train_idx, val_idx) in enumerate(cv_splits):
    X_tr_f, y_tr_f = X_tree_train.iloc[train_idx], y[train_idx]
    X_va_f, y_va_f = X_tree_train.iloc[val_idx], y[val_idx]
    
    dtrain = lgb.Dataset(X_tr_f, label=y_tr_f)
    dval   = lgb.Dataset(X_va_f, label=y_va_f, reference=dtrain)
    
    model_tuned = lgb.train(
        full_tuned_params,
        dtrain,
        valid_sets=[dval],
        num_boost_round=1500,
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    
    val_preds = model_tuned.predict(X_va_f)
    oof_lgb_tuned[val_idx] = val_preds
    test_lgb_tuned += model_tuned.predict(X_tree_test) / N_FOLDS
    print(f"Tuned LightGBM Fold {fold + 1} ROC-AUC: {roc_auc_score(y_va_f, val_preds):.5f}")

overall_tuned_auc = roc_auc_score(y, oof_lgb_tuned)
print(f"\n>>> OVERALL Tuned LightGBM OOF ROC-AUC: {overall_tuned_auc:.5f} <<<")
np.save('oof_lgb_tuned.npy', oof_lgb_tuned)
np.save('test_lgb_tuned.npy', test_lgb_tuned)
print("Saved oof_lgb_tuned.npy and test_lgb_tuned.npy")

In [ ]:
# =============================================================================
# 8. Out-of-Fold (OOF) Evaluation, Diagnostic Curves & Confusion Matrix
# =============================================================================
# Evaluate metrics on full OOF predictions across the entire training dataset
primary_oof = oof_lgb_tuned if 'oof_lgb_tuned' in locals() else oof_lgb

final_oof_auc = roc_auc_score(y, primary_oof)
final_oof_pr  = average_precision_score(y, primary_oof)

print(f"Overall Out-of-Fold ROC-AUC : {final_oof_auc:.5f}")
print(f"Overall Out-of-Fold PR-AUC  : {final_oof_pr:.5f}")

# Classification diagnostic using target baseline threshold (since submission is probability-scored)
baseline_threshold = float(y.mean())
oof_binary_preds = (primary_oof >= baseline_threshold).astype(int)
print(f"\n--- Classification Report (At Prior Rate Threshold = {baseline_threshold:.4f}) ---")
print(classification_report(y, oof_binary_preds, digits=4))

# Visualizations: ROC Curve, PR Curve & Confusion Matrix
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. ROC Curve
fpr, tpr, _ = roc_curve(y, primary_oof)
axes[0].plot(fpr, tpr, color='darkorange', lw=2, label=f'OOF ROC (AUC = {final_oof_auc:.4f})')
axes[0].plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
axes[0].set_title('Out-of-Fold ROC Curve')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].legend(loc='lower right')

# 2. Precision-Recall Curve
precs, recs, _ = precision_recall_curve(y, primary_oof)
axes[1].plot(recs, precs, color='green', lw=2, label=f'OOF PR (AUC = {final_oof_pr:.4f})')
axes[1].set_title('Out-of-Fold Precision-Recall Curve')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right')

# 3. Confusion Matrix
cm = confusion_matrix(y, oof_binary_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[2],
            xticklabels=['Legit (0)', 'Fraud (1)'],
            yticklabels=['Legit (0)', 'Fraud (1)'])
axes[2].set_title(f'Confusion Matrix (Threshold: {baseline_threshold:.4f})')
axes[2].set_ylabel('Actual')
axes[2].set_xlabel('Predicted')

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# 9. Multi-Fold & Multi-Seed Test Predictions Verification
# =============================================================================
# Verify all test prediction arrays have been averaged across folds/seeds and match test dimensions
print(f"Test set row count: {len(test_df)}")

saved_models = ['lgb', 'lgb_tuned', 'xgb', 'cat', 'mlp']
for name in saved_models:
    if os.path.exists(f'test_{name}.npy'):
        arr = np.load(f'test_{name}.npy')
        print(f"  test_{name}.npy  -> shape: {arr.shape} | mean: {arr.mean():.4f} | min: {arr.min():.4f} | max: {arr.max():.4f}")

# =============================================================================
# 10. Multi-Model OOF Ensembling (Constrained Blending, Stacking & Rank Averaging)
# =============================================================================
This section loads all 5-Fold OOF predictions from:
- **LightGBM** (Baseline & Tuned)
- **XGBoost** (`hist`)
- **CatBoost** (Native Categoricals)
- **PyTorch MLP** (Embeddings & Residuals across multiple seeds)

We construct and compare three competition-grade ensembling methods:
1. **Constrained Weighted Blend**: Non-negative weights summing to 1, optimized via `scipy.optimize.minimize` (SLSQP) on OOF ROC-AUC.
2. **Meta-Learner Stacking**: Logistic Regression trained with secondary 5-Fold CV on OOF probabilities.
3. **Rank Averaging**: Combining rank percentiles to eliminate calibration discrepancies.

In [ ]:
# =============================================================================
# 10. Ensemble Pipeline: Constrained Blending, Stacking & Rank Averaging
# =============================================================================
# --- 10.1 Load all available OOF and Test predictions ---
model_registry = {}
candidate_names = ['lgb_tuned', 'lgb', 'xgb', 'cat', 'mlp']

for name in candidate_names:
    oof_file = f'oof_{name}.npy'
    test_file = f'test_{name}.npy'
    if os.path.exists(oof_file) and os.path.exists(test_file):
        oof_arr = np.load(oof_file)
        test_arr = np.load(test_file)
        score = roc_auc_score(y, oof_arr)
        model_registry[name] = {'oof': oof_arr, 'test': test_arr, 'auc': score}
        print(f"Registered {name:10s} | Single Model OOF ROC-AUC: {score:.5f}")

active_names = list(model_registry.keys())
oof_matrix = np.column_stack([model_registry[n]['oof'] for n in active_names])
test_matrix = np.column_stack([model_registry[n]['test'] for n in active_names])
M = len(active_names)

# --- 10.2 Strategy A: Constrained Non-Negative Weighted Blending (scipy.optimize) ---
def auc_loss(weights):
    weights = np.array(weights)
    blend = np.dot(oof_matrix, weights)
    return -roc_auc_score(y, blend)

init_weights = np.ones(M) / M
bounds = [(0.0, 1.0) for _ in range(M)]
constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1.0})

opt_res = minimize(auc_loss, init_weights, method='SLSQP', bounds=bounds, constraints=constraints)
opt_weights = opt_res.x

oof_blend = np.dot(oof_matrix, opt_weights)
blend_auc = roc_auc_score(y, oof_blend)
test_blend = np.dot(test_matrix, opt_weights)

print(f"\n--- [Strategy A] Constrained Optimal Blend ---")
print(f"OOF ROC-AUC: {blend_auc:.5f}")
for n, w in zip(active_names, opt_weights):
    print(f"  Weight {n:10s}: {w:.4f}")

# --- 10.3 Strategy B: Logistic Regression Stacker with Internal 5-Fold CV ---
oof_stack = np.zeros(len(train_df), dtype=np.float32)
test_stack = np.zeros(len(test_df), dtype=np.float32)

for tr_idx, va_idx in skf.split(oof_matrix, y):
    meta_clf = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
    meta_clf.fit(oof_matrix[tr_idx], y[tr_idx])
    oof_stack[va_idx] = meta_clf.predict_proba(oof_matrix[va_idx])[:, 1]
    test_stack += meta_clf.predict_proba(test_matrix)[:, 1] / N_FOLDS

stack_auc = roc_auc_score(y, oof_stack)
print(f"\n--- [Strategy B] Logistic Regression Stacker ---")
print(f"OOF ROC-AUC: {stack_auc:.5f}")

# --- 10.4 Strategy C: Kaggle Rank Averaging ---
oof_ranks = np.column_stack([rankdata(oof_matrix[:, i]) / len(train_df) for i in range(M)])
test_ranks = np.column_stack([rankdata(test_matrix[:, i]) / len(test_df) for i in range(M)])

oof_rank_avg = np.dot(oof_ranks, opt_weights)
test_rank_avg = np.dot(test_ranks, opt_weights)
rank_auc = roc_auc_score(y, oof_rank_avg)

print(f"\n--- [Strategy C] Rank Averaged Ensemble ---")
print(f"OOF ROC-AUC: {rank_auc:.5f}")

# --- 10.5 Pick the Best Performing Ensemble Strategy ---
candidates = [
    ('Weighted Blend', blend_auc, test_blend, oof_blend),
    ('Stacker', stack_auc, test_stack, oof_stack),
    ('Rank Average', rank_auc, test_rank_avg, oof_rank_avg)
]
candidates.sort(key=lambda x: x[1], reverse=True)
best_name, best_score, best_test_preds, best_oof_preds = candidates[0]

print(f"\n>>> WINNING ENSEMBLE STRATEGY: {best_name} with OOF ROC-AUC: {best_score:.5f} <<<")

# Normalize probabilities strictly to [0, 1]
final_probs = np.clip(best_test_preds, 0.0, 1.0)

# --- 10.6 Generate submission.csv matching sample_submission.csv format ---
submission = pd.DataFrame({
    'id': test_ids,
    'is_fraud': final_probs
})

# Guarantee identical row order and column names
submission = sample_sub[['id']].merge(submission, on='id', how='left')
output_path = 'submission.csv'
submission.to_csv(output_path, index=False)
print(f"\nSaved final submission to: {output_path}")

# --- 10.7 Distribution Sanity Check (OOF vs Test Predictions) ---
print("\n--- Sanity Check: Prediction Distribution Comparison ---")
dist_check = pd.DataFrame({
    'Metric': ['Mean', 'Std', 'Min', '25%', '50%', '75%', '99%', 'Max'],
    'OOF Predictions': [
        best_oof_preds.mean(), best_oof_preds.std(), best_oof_preds.min(),
        np.percentile(best_oof_preds, 25), np.percentile(best_oof_preds, 50),
        np.percentile(best_oof_preds, 75), np.percentile(best_oof_preds, 99),
        best_oof_preds.max()
    ],
    'Test Predictions': [
        final_probs.mean(), final_probs.std(), final_probs.min(),
        np.percentile(final_probs, 25), np.percentile(final_probs, 50),
        np.percentile(final_probs, 75), np.percentile(final_probs, 99),
        final_probs.max()
    ]
})
print(dist_check.round(5).to_string(index=False))

# --- 10.8 Summary Performance Leaderboard Table ---
print("\n" + "="*60)
print("         FINAL CROSS-VALIDATION LEADERBOARD         ")
print("="*60)
leaderboard = []
for n in active_names:
    leaderboard.append({'Model / Strategy': f'Single: {n}', 'OOF ROC-AUC': model_registry[n]['auc']})
leaderboard.append({'Model / Strategy': 'Ensemble: Weighted Blend', 'OOF ROC-AUC': blend_auc})
leaderboard.append({'Model / Strategy': 'Ensemble: Stacker (LogisticReg)', 'OOF ROC-AUC': stack_auc})
leaderboard.append({'Model / Strategy': 'Ensemble: Rank Averaging', 'OOF ROC-AUC': rank_auc})

lb_df = pd.DataFrame(leaderboard).sort_values(by='OOF ROC-AUC', ascending=False).reset_index(drop=True)
best_single = max([model_registry[n]['auc'] for n in active_names])
lb_df['Gain vs Best Single'] = (lb_df['OOF ROC-AUC'] - best_single).map(lambda g: f"+{g:.5f}" if g > 0 else f"{g:.5f}")
print(lb_df.to_string(index=False))
print("="*60)

print("\nFirst 5 rows of submission.csv:")
print(submission.head())